In [ ]:
import geemap
import ee 
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
Map = geemap.Map()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')

In [ ]:
# 1. Load and Filter Sentinel-2 Surface Reflectance (2026)
# We use the Harmonized collection for consistency
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2026-01-01', '2026-03-20') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

In [ ]:
#Visualising this: 
vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B4", "B3", "B2"],
}
Map.centerObject(roi, 10)
Map.addLayer(s2, vis, "Sentinel-2")
Map


Program to Check NDBI 

In [ ]:
# NDBI: Built-up/Urban Index
def get_ndbi(image):
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return ndbi

# --- Map 1: NDBI (Built-up / Urban) ---
Map_NDBI = geemap.Map()
Map_NDBI.centerObject(roi, 10)
ndbi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'orange', 'red']}

Map_NDBI.addLayer(get_ndbi(s2), ndbi_params, 'NDBI (Urban)')
Map_NDBI.add_colorbar(vis_params=ndbi_params, label="NDBI Value", orientation="horizontal")
print("Displaying NDBI Map...")
display(Map_NDBI)

Program to check MNDWI

In [ ]:
# MNDWI: Water Index
def get_mndwi(image):
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return mndwi

# --- Map 2: MNDWI (Water Index) ---
Map_MNDWI = geemap.Map()
Map_MNDWI.centerObject(roi, 10)
mndwi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'lightblue', 'blue']}

Map_MNDWI.addLayer(get_mndwi(s2), mndwi_params, 'MNDWI (Water)')
Map_MNDWI.add_colorbar(vis_params=mndwi_params, label="MNDWI Value", orientation="horizontal")
print("Displaying MNDWI Map...")
display(Map_MNDWI)

Program to check SAVI

In [ ]:
# SAVI: Soil Adjusted Vegetation Index
def get_savi(image):
    savi = image.expression(
        '((NIR - RED) * 1.5) / (NIR + RED + 0.5)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    return savi

# --- Map 3: SAVI (Soil Adjusted Vegetation) ---
Map_SAVI = geemap.Map()
Map_SAVI.centerObject(roi, 10)
savi_params = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}

Map_SAVI.addLayer(get_savi(s2), savi_params, 'SAVI (Vegetation)')
Map_SAVI.add_colorbar(vis_params=savi_params, label="SAVI Value", orientation="horizontal")
print("Displaying SAVI Map...")
display(Map_SAVI)

Understanding the Results
NDBI: In the red/orange areas, you are likely looking at residential or commercial zones.

MNDWI: Look for the deep blue pixels; those will be your lakes, rivers, or reservoirs.

SAVI: The vibrant green represents healthy biomass. If you compare this to a standard NDVI, you'll notice it handles the edges of fields (where soil is visible) much more accurately.

In [ ]:
# 2. Define Index Functions
def calculate_indices(image):
    # Band aliases for Sentinel-2
    # B11=SWIR1, B8=NIR, B3=Green, B4=Red
    res = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('NDBI')
    
    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)', {
            'Green': image.select('B3'),
            'SWIR1': image.select('B11')
        }).rename('MNDWI')
    
    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)', {
            'NIR': image.select('B8'),
            'Red': image.select('B4')
        }).rename('SAVI')
        
    return image.addBands([res, mndwi, savi])

# Apply the indices
processed_img = calculate_indices(s2)

Program to check and fine-tune the logic of Urban Build-up measurement

In [ ]:
# 3. Create the "Clean" Built-up Mask (The Logic)
# Thresholds: 
# NDBI > 0 (Built-up typically positive)
# MNDWI < 0 (Exclude water/moist salt pans)
# SAVI < 0.2 (Exclude vegetation)
built_up_mask = processed_img.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.15)', {
        'NDBI': processed_img.select('NDBI'),
        'MNDWI': processed_img.select('MNDWI'),
        'SAVI': processed_img.select('SAVI')
    }).rename('BuiltUp')

In [ ]:
# 4. Visualization
Map = geemap.Map()
Map.centerObject(roi, 11)

# True Color for Reference
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2',], 'min': 0, 'max': 3000}, 'S2 True Color (2026)')

# The Result: 1 = White (Built-up), 0 = Black (Everything Else)
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'red']}, 'Clean Built-up Layer')

Map